<a id="gh200-gpu"></a>
# 01-2. GH200 — GPU 메모리 경로와 프로파일링

**세션:** 14:00–14:30  
**학습 목표:** 같은 벡터 덧셈을 명시적 복사(`cudaMemcpy`), CUDA 통합 메모리(`cudaMallocManaged`), 시스템 할당 메모리(`new`)로 실행하고, Nsight Systems의 실행 기록과 `nvbandwidth` 측정값을 해석합니다.

시스템 할당 메모리의 GPU 직접 접근 여부는 런타임 속성으로 확인합니다. 지원되지 않는 경우 프로그램은 `SKIP`을 출력하고 정상 종료하며, 나머지 두 메모리 방식으로 실습을 계속합니다.


## 1. 과정 폴더, Hopper GPU, 프로파일링 도구 확인

통합 컨테이너 이미지에 CUDA 컴파일러, Nsight Systems, nvbandwidth가 준비돼 있는지 확인합니다.


In [ ]:
from pathlib import Path
import sys

launch_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "gh200" / "notebook_utils.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("labs/gh200/notebook_utils.py를 찾지 못했습니다.")

LAB_DIR = REPO_ROOT / "labs" / "gh200"
WORK_DIR = REPO_ROOT / "work" / "gh200"
BIN_DIR = WORK_DIR / "bin"
PROFILE_DIR = WORK_DIR / "profiles"
BIN_DIR.mkdir(parents=True, exist_ok=True)
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from notebook_utils import print_tool_status, read_image_manifest, run, system_summary

required_tools = ("nvcc", "nsys", "nvbandwidth")
if not print_tool_status(required_tools):
    raise EnvironmentError("필수 도구가 없습니다. 계산 노드에서 설치하지 말고 강사에게 알리세요.")

summary = system_summary()
for key, value in summary.items():
    print(f"{key:14s}: {value}")
if str(summary["architecture"]).lower() not in {"aarch64", "arm64"}:
    raise EnvironmentError(f"ARM64 환경이 아닙니다: {summary['architecture']}")
if "GH200" not in str(summary.get("gpu", "")).upper():
    raise EnvironmentError(f"GH200을 확인하지 못했습니다: {summary.get('gpu')}")
if read_image_manifest() is None:
    raise FileNotFoundError("통합 image manifest를 찾지 못했습니다: /etc/ksc2026-image.json")
print("Image manifest: PASS")


## 2. 세 가지 CUDA 메모리 방식으로 빌드

세 소스 코드는 같은 벡터 덧셈을 수행하지만 메모리의 할당과 이동 방식이 다릅니다.

| 방식 | 메모리 할당과 이동 | 확인 항목 |
|---|---|---|
| 명시적 복사(Explicit copy) | 호스트와 디바이스 메모리를 따로 할당하고 `cudaMemcpy`로 복사 | 데이터 이동 시점을 코드에 직접 지정 |
| 통합 메모리(Unified Memory) | `cudaMallocManaged`로 할당하고 사전 이동(prefetch) 수행 | 하나의 포인터를 사용하며 필요에 따라 데이터가 이동 |
| 시스템 할당 메모리 | `new`로 만든 pageable system memory의 포인터를 커널에 전달 | GPU의 직접 접근 지원 여부와 실제 일관성 경로 |

시스템 할당 메모리는 세 번째 **할당 방식**입니다. ATS와 HMM은 GPU가 시스템 할당 메모리에 접근할 때 선택될 수 있는 **일관성 경로**이며 별도의 할당 API가 아닙니다. 프로그램은 `cudaDevAttrPageableMemoryAccess`와 `cudaDevAttrPageableMemoryAccessUsesHostPageTables`를 확인해 지원 여부와 경로를 판별합니다. CPU 페이지 테이블을 GPU가 사용하면 ATS 기반 하드웨어 일관성 경로이며, HMM은 ATS가 없는 지원 환경에서 Linux가 제공하는 소프트웨어 일관성 경로입니다.

직접 접근을 지원하지 않는 환경에서는 `SKIP`이 표시됩니다. 이 경우 시스템 할당 메모리의 성능값은 기록하지 않습니다.


In [ ]:
CUDA_SOURCE_DIR = LAB_DIR / "cuda_memory"
memory_sources = {
    "explicit": CUDA_SOURCE_DIR / "explicit.cu",
    "managed": CUDA_SOURCE_DIR / "managed.cu",
    # 기존 파일명은 hmm.cu이지만, 실행 결과는 ATS와 HMM을 구분합니다.
    "system": CUDA_SOURCE_DIR / "hmm.cu",
}

for name, source in memory_sources.items():
    if not source.is_file():
        raise FileNotFoundError(source)
    output = BIN_DIR / f"cuda-{name}"
    run(
        ["nvcc", "-O3", "-std=c++17", "-arch=sm_90", str(source), "-o", str(output)],
        timeout=600,
    )
print("CUDA 빌드 결과: PASS")


In [ ]:
MEMORY_RESULTS = {}
for name in ("explicit", "managed", "system"):
    completed = run([BIN_DIR / f"cuda-{name}"], timeout=300)
    MEMORY_RESULTS[name] = completed.stdout.strip()

SYSTEM_MEMORY_SUPPORTED = "SKIP" not in MEMORY_RESULTS["system"]
if SYSTEM_MEMORY_SUPPORTED:
    system_path = (
        "ATS / hardware-coherent"
        if "ATS/hardware-coherent" in MEMORY_RESULTS["system"]
        else "HMM / software-coherent"
    )
    print(f"\n시스템 할당 메모리 경로: {system_path}")
else:
    print("\n시스템 할당 메모리 경로: SKIP (기능 미지원)")


## 3. Nsight Systems로 CUDA 실행 기록 수집

명시적 복사와 통합 메모리는 항상 프로파일링하고, 시스템 할당 메모리는 GPU가 직접 접근할 수 있을 때만 프로파일링합니다. 보고서는 `work/gh200/profiles/`에 저장합니다. 노트북에는 CUDA API와 커널 실행 시간의 CLI 집계표가 표시됩니다. 전체 실행 타임라인은 생성된 `.nsys-rep`를 Nsight Systems UI로 열어 강사 화면에서 함께 확인합니다.


In [ ]:
PROFILE_TARGETS = ["explicit", "managed"] + (
    ["system"] if SYSTEM_MEMORY_SUPPORTED else []
)
NSYS_REPORTS = {}

for name in PROFILE_TARGETS:
    profile_base = PROFILE_DIR / f"cuda-{name}"
    report_path = profile_base.with_suffix(".nsys-rep")
    run(
        [
            "nsys",
            "profile",
            "--trace=cuda,osrt",
            "--sample=none",
            "--cuda-memory-usage=true",
            "--force-overwrite=true",
            "--output",
            str(profile_base),
            str(BIN_DIR / f"cuda-{name}"),
        ],
        timeout=600,
    )
    if not report_path.is_file():
        raise FileNotFoundError(report_path)
    NSYS_REPORTS[name] = report_path
    run(
        ["nsys", "stats", "--report", "cuda_api_sum,cuda_gpu_kern_sum", str(report_path)],
        timeout=600,
    )

if not SYSTEM_MEMORY_SUPPORTED:
    print(
        "System-memory profile: SKIP — "
        "cudaDevAttrPageableMemoryAccess=false"
    )


### CLI 집계표와 `.nsys-rep` 타임라인을 살펴볼 질문

- 명시적 복사 방식에는 어떤 `cudaMemcpy` 호출이 보이나요?
- 통합 메모리에서 사전 이동을 수행한 뒤, 커널 실행 전에 추가 데이터 이동이 관찰되나요?
- `nsys`가 보고한 API 실행 시간과 프로그램 전체 실행 시간이 다른 이유는 무엇인가요?
- 시스템 할당 메모리 경로가 ATS인지 HMM인지 프로그램 출력에서 확인했나요?
- 이 경로가 `SKIP`이면 어떤 기능을 지원하지 않는 것일까요? 다른 메모리 방식의 결과만으로 성능을 추정하면 왜 안 될까요?


## 4. nvbandwidth로 호스트↔디바이스 전송 대역폭 측정

전체 테스트 모음 대신 수업에서 다룰 두 가지 복사 엔진 테스트만 실행합니다. 여기서 측정한 전송 대역폭은 앞의 벡터 덧셈 프로그램 전체 실행 시간과 서로 다른 지표입니다.


In [ ]:
NVBANDWIDTH_COMMAND = [
    "nvbandwidth",
    "-t",
    "host_to_device_memcpy_ce",
    "device_to_host_memcpy_ce",
]
NVBANDWIDTH_RESULT = run(NVBANDWIDTH_COMMAND, timeout=900)


## 5. 결과 정리와 완료 확인

| 확인 항목 | 해석 |
|---|---|
| 벡터 덧셈 결과 | 지원되는 메모리 방식이 같은 계산 결과를 냈는지 확인 |
| 시스템 할당 메모리 상태 | GPU 직접 접근 지원 여부와 ATS·HMM 경로 확인 |
| Nsight Systems 집계표 | CUDA API와 GPU 커널에 사용된 시간 확인 |
| `.nsys-rep` | API·메모리 작업·커널의 실행 순서 확인 |
| nvbandwidth | 선택한 호스트↔디바이스 복사 경로의 대역폭 확인 |

측정값은 현재 프로그램과 Slurm 할당 조건에 대한 결과입니다. 특정 메모리 방식이 모든 작업에서 더 빠르다는 결론이나, 다른 노드의 동시 부하까지 반영한 공식 벤치마크를 뜻하지 않습니다.

**완료 체크리스트**

- [ ] CUDA 예제 세 개를 빌드했습니다.
- [ ] 지원되는 메모리 방식의 계산 결과를 확인했습니다.
- [ ] 시스템 할당 메모리의 지원 상태와 일관성 경로를 기록했습니다.
- [ ] 지원되는 경로별 `.nsys-rep`를 생성했습니다.
- [ ] nvbandwidth의 호스트→디바이스와 디바이스→호스트 측정값을 확인했습니다.

다음 세션은 [PhysicsNeMo 발사체 운동 PINN](../02_PhysicsNeMo/01_Projectile_PINN.ipynb)입니다.


---

## 출처와 라이선스

이 실습은 KSC 2026을 위해 별도로 작성한 소스 코드를 사용합니다. CUDA, Nsight Systems, nvbandwidth에는 각 원저작물의 라이선스와 고지가 적용됩니다.
